In [1]:
# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

from pyswarm import pso

In [2]:
# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv("NIFTY_5_Years.csv")

df.columns = df.columns.str.strip()
df['Date'] = pd.to_datetime(df['Date'])

df.set_index('Date', inplace=True)
df = df.sort_index()

data = df[['Close']]

# Train-Test Split
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

# Scaling
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

print("Train:", len(train), "Test:", len(test))

Train: 992 Test: 248


In [3]:
# ============================================================
# SEQUENCE FUNCTION
# ============================================================

def create_sequences(data, time_steps):
    X, y = [], []
    
    for i in range(len(data) - time_steps):
        X.append(data[i:i+time_steps])
        y.append(data[i+time_steps])
    
    return np.array(X), np.array(y)

In [4]:
# ============================================================
# OBJECTIVE FUNCTION
# ============================================================

def objective_function(params):
    try:
        p, d, q, P, D, Q, m = map(int, params)

        model = SARIMAX(train['Close'],
                        order=(p,d,q),
                        seasonal_order=(P,D,Q,m))

        result = model.fit(disp=False)

        forecast = result.forecast(steps=len(test))
        actual = test['Close'].values

        rmse = np.sqrt(mean_squared_error(actual, forecast))

        return rmse

    except:
        return 1e10

In [5]:
# ============================================================
# PARAMETER BOUNDS
# ============================================================

lb = [1, 0, 1,   0, 0, 0,   5, 32]    # lower bounds
ub = [5, 1, 5,   2, 1, 2,  21, 128]   # upper bounds

In [6]:
# ============================================================
# RUN PSO
# ============================================================

best_params, best_score = pso(
    objective_function,
    lb,
    ub,
    swarmsize=40,
    maxiter=20
)

print("\n===== BEST RESULT =====")
print("Best Parameters:", best_params)
print("Best RMSE:", best_score)

Stopping search: maximum iterations reached --> 20

===== BEST RESULT =====
Best Parameters: [ 2.09583952  0.45500655  1.21961181  1.11175377  0.69725058  1.40198539
 13.89156039 77.73311576]
Best RMSE: 10000000000.0


In [7]:
# ============================================================
# TRAIN FINAL MODEL WITH BEST PARAMS
# ============================================================

p, d, q, P, D, Q, m, units = map(int, best_params)

print("Using Best Params:", p, d, q, P, D, Q, m, units)

Using Best Params: 2 0 1 1 0 1 13 77
